<a href="https://colab.research.google.com/github/Camisrad/AlphafoldHGTpipeline/blob/main/alphafold_hgt_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AlphaFold HGT Structural Validation Pipeline
**Mathur Lab | Cameron Liggett**

Validates phage→bacteria HGT candidate pairs with structural evidence.

| Step | Tool | What it does |
|------|------|--------------|
| 1 | AlphaFold EBI DB | Download precomputed structures (fast) |
| 2 | NCBI Entrez | Fetch sequences for proteins not in AlphaFold DB |
| 3 | ESMFold API | Predict structures for sequences not in DB |
| 4 | TMalign | TM-score + RMSD for each phage/bacteria pair |
| 5 | Foldseek | Search phage proteins against AlphaFold DB + PDB |
| 6 | pandas | Manuscript-ready results table (CSV) |
| 7 | py3Dmol | 3D superposition visualization for poster |

**TM-score thresholds:** < 0.17 = random | > 0.50 = same fold | > 0.70 = very similar

Run all cells top-to-bottom. Outputs saved to `structures/`, `sequences/`, `results/`.

In [ ]:
# @title Step 0: Install dependencies and compile TMalign
!pip install -q biopython requests pandas py3Dmol

import os, subprocess

# Compile TMalign from source (runs on Colab's Linux environment)
if not os.path.exists('./TMalign'):
    print('Downloading and compiling TMalign...')
    r1 = subprocess.run(['wget', '-q', 'https://zhanggroup.org/TM-align/TMalign.cpp'],
                        capture_output=True)
    if r1.returncode == 0:
        r2 = subprocess.run(['g++', '-O3', '-o', 'TMalign', 'TMalign.cpp'],
                            capture_output=True)
        if r2.returncode == 0:
            os.chmod('./TMalign', 0o755)
            print('TMalign compiled successfully.')
        else:
            print('TMalign compilation failed.')
            print('TM-score step will be skipped.')
            print('Use https://zhanggroup.org/US-align/ manually instead.')
    else:
        print('TMalign download failed — check network.')
else:
    print('TMalign already compiled.')

for d in ['sequences', 'structures', 'results']:
    os.makedirs(d, exist_ok=True)

print('Setup complete.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 29.3 MB/s eta 0:00:00
TMalign compiled successfully.
Setup complete.


In [ ]:
# @title Step 0b: Configuration

NCBI_EMAIL    = 'liggettcam@gmail.com'  # @param {type:"string"}
RUN_UNCERTAIN = False  # @param {type:"boolean"}

# Confirmed positive pairs (phage_accession, bacteria_accession, row_label, blast_note)
CONFIRMED_PAIRS = [
    ('YP_009168097.1', 'WP_000994792.1', 'Row 1',  'Clean positive'),
    ('YP_009168179.1', 'KLD66510.1',     'Row 2',  'Positive — reverse search, 52% query / 77% identity'),
    ('YP_009168137.1', 'EDS05563',       'Row 3',  'Positive — non-NR database, 100%/100%'),
    ('YP_009168122.1', 'EFQ7096440.1',  'Row 10', 'Clean first result'),
    ('YP_009168121.1', 'WP_000062360.1','Row 11', 'Clean first result'),
]

UNCERTAIN_PAIRS = [
    ('YP_009908453.1', 'WP_000438489.1', 'Row 28', 'Uncertain positive'),
    ('YP_009908449.1', 'WP_001221210.1', 'Row 30', 'Uncertain positive'),
    ('NP_859243.1',    'AFJ28224.1',     'Row 35', 'Uncertain positive'),
]

PAIRS    = CONFIRMED_PAIRS + (UNCERTAIN_PAIRS if RUN_UNCERTAIN else [])
ALL_ACCS = list(dict.fromkeys(acc for p in PAIRS for acc in (p[0], p[1])))

print(f'{len(PAIRS)} pairs to process | {len(ALL_ACCS)} unique accessions')
print()
for p in PAIRS:
    print(f'  {p[2]:7s}  {p[0]:20s}  vs  {p[1]:20s}  [{p[3]}]')

5 pairs to process | 10 unique accessions

  Row 1    YP_009168097.1        vs  WP_000994792.1        [Clean positive]
  Row 2    YP_009168179.1        vs  KLD66510.1            [Positive — reverse search, 52% query / 77% identity]
  Row 3    YP_009168137.1        vs  EDS05563              [Positive — non-NR database, 100%/100%]
  Row 10   YP_009168122.1        vs  EFQ7096440.1          [Clean first result]
  Row 11   YP_009168121.1        vs  WP_000062360.1        [Clean first result]


In [ ]:
# Helper functions (run this cell before Steps 1-7)

import requests, time, os, subprocess
from Bio import Entrez, SeqIO
import pandas as pd

Entrez.email = NCBI_EMAIL

REFSEQ_PREFIXES = ('YP_', 'WP_', 'NP_', 'XP_', 'ZP_', 'AP_')

def slug(acc):
    return acc.replace('.', '_')

def struct_path(acc):
    return f'structures/{slug(acc)}.pdb'

# ── NCBI ─────────────────────────────────────────────────────────────────────

def fetch_ncbi(acc):
    handle = Entrez.efetch(db='protein', id=acc, rettype='fasta', retmode='text')
    try:
        rec = SeqIO.read(handle, 'fasta')
    finally:
        handle.close()
    return str(rec.seq), rec.description

def save_fasta(acc, seq):
    path = f'sequences/{slug(acc)}.fasta'
    with open(path, 'w') as f:
        f.write(f'>{acc}\n{seq}\n')
    return path

# ── UniProt ID mapping ────────────────────────────────────────────────────────

def map_to_uniprot(acc):
    base    = acc.split('.')[0]
    from_db = 'RefSeq_Protein' if any(acc.startswith(p) for p in REFSEQ_PREFIXES) else 'EMBL'
    r = requests.post('https://rest.uniprot.org/idmapping/run',
                      data={'from': from_db, 'to': 'UniProtKB', 'ids': base})
    if not r.ok:
        return None
    job_id = r.json()['jobId']
    for _ in range(20):
        time.sleep(3)
        s = requests.get(f'https://rest.uniprot.org/idmapping/status/{job_id}').json()
        if 'results' in s:
            return s['results'][0]['to']['primaryAccession'] if s['results'] else None
        if 'failedIds' in s or s.get('status') == 'FAILURE':
            return None
    return None

# ── AlphaFold DB ──────────────────────────────────────────────────────────────

def fetch_alphafold_pdb(uniprot_id, acc):
    r = requests.get(f'https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}')
    if r.status_code != 200:
        return None
    pdb_r = requests.get(r.json()[0]['pdbUrl'])
    pdb_r.raise_for_status()
    path = struct_path(acc)
    with open(path, 'w') as f:
        f.write(pdb_r.text)
    return path

# ── ESMFold ───────────────────────────────────────────────────────────────────

def run_esmfold(seq, acc):
    r = requests.post(
        'https://api.esmatlas.com/foldSequence/v1/pdb/',
        headers={'Content-Type': 'application/x-www-form-urlencoded'},
        data=seq,
        timeout=300
    )
    r.raise_for_status()
    path = struct_path(acc)
    with open(path, 'w') as f:
        f.write(r.text)
    return path

# ── TMalign ───────────────────────────────────────────────────────────────────

def run_tmalign(pdb1, pdb2, out_prefix=None):
    cmd = ['./TMalign', pdb1, pdb2]
    if out_prefix:
        cmd += ['-o', out_prefix]  # saves superimposed PDB for visualization
    res = subprocess.run(cmd, capture_output=True, text=True)
    tm1 = tm2 = rmsd = aln_len = None
    for line in res.stdout.split('\n'):
        if 'TM-score=' in line and 'Chain_1' in line:
            try: tm1 = float(line.split('TM-score=')[1].split()[0])
            except: pass
        if 'TM-score=' in line and 'Chain_2' in line:
            try: tm2 = float(line.split('TM-score=')[1].split()[0])
            except: pass
        if 'Aligned length=' in line and 'RMSD=' in line:
            try: rmsd = float(line.split('RMSD=')[1].strip().split(',')[0])
            except: pass
            try: aln_len = int(line.split('Aligned length=')[1].strip().split(',')[0])
            except: pass
    valid = [t for t in [tm1, tm2] if t is not None]
    tm = max(valid) if valid else None
    return tm, rmsd, aln_len

def tm_label(tm):
    if tm is None:  return 'N/A — structure missing or TMalign unavailable'
    if tm >= 0.70:  return 'STRONG structural similarity — supports HGT'
    if tm >= 0.50:  return 'MODERATE structural similarity — consistent with HGT'
    if tm >= 0.17:  return 'WEAK structural similarity — inconclusive'
    return 'RANDOM similarity — no structural relationship'

# ── Foldseek ─────────────────────────────────────────────────────────────────

def run_foldseek(pdb_path, databases=None):
    if databases is None:
        databases = ['afdb50', 'pdb100']
    with open(pdb_path, 'rb') as f:
        resp = requests.post(
            'https://search.foldseek.com/api/ticket',
            files={'q': f},
            data={'database[]': databases, 'mode': 'tmalign'}
        )
    if not resp.ok:
        return []
    ticket = resp.json()['id']
    for _ in range(60):
        time.sleep(5)
        s = requests.get(f'https://search.foldseek.com/api/ticket/{ticket}').json()
        if s.get('status') == 'COMPLETE':
            break
        if s.get('status') == 'ERROR':
            return []
    res = requests.get(f'https://search.foldseek.com/api/result/{ticket}/0').json()
    hits = []
    for db_result in res.get('results', []):
        for hit in db_result.get('alignments', [])[:5]:
            hits.append({
                'db':       db_result.get('db', ''),
                'target':   hit.get('target', ''),
                'tm_score': round(hit.get('prob', 0), 4),
                'evalue':   hit.get('eval', 0),
                'organism': hit.get('taxName', 'N/A'),
            })
    return sorted(hits, key=lambda h: h['tm_score'], reverse=True)

print('All helper functions loaded.')

All helper functions loaded.


In [ ]:
# Steps 1-3: For each accession — check AlphaFold DB, fetch sequence, predict if needed

structure_source = {}  # acc -> 'AlphaFold' | 'ESMFold' | 'FAILED'
protein_names    = {}  # acc -> description
sequences        = {}  # acc -> sequence string

SEP = '=' * 65
print(SEP)
print('STEPS 1-3: FETCHING SEQUENCES AND STRUCTURES')
print(SEP)

for acc in ALL_ACCS:
    print(f'\n[{acc}]')

    # Skip if structure already exists on disk
    if os.path.exists(struct_path(acc)):
        kb = os.path.getsize(struct_path(acc)) / 1024
        src = structure_source.get(acc, 'cached')
        print(f'  Already have structure ({kb:.0f} KB) [{src}] — skipping.')
        continue

    # Step 2: Fetch sequence from NCBI
    try:
        seq, desc = fetch_ncbi(acc)
        sequences[acc]     = seq
        protein_names[acc] = desc
        save_fasta(acc, seq)
        print(f'  NCBI: {len(seq)} aa — {desc[:70]}')
    except Exception as e:
        print(f'  NCBI FAILED: {e}')
        structure_source[acc] = 'FAILED'
        continue

    # Step 1: Check AlphaFold DB via UniProt mapping
    print('  AlphaFold DB: mapping to UniProt...', end=' ', flush=True)
    uniprot = map_to_uniprot(acc)
    if uniprot:
        print(f'UniProt={uniprot}, checking DB...', end=' ', flush=True)
        af_path = fetch_alphafold_pdb(uniprot, acc)
        if af_path:
            structure_source[acc] = 'AlphaFold'
            print('downloaded.')
            continue
        else:
            print('not in DB.', end=' ')
    else:
        print('no UniProt mapping.', end=' ')

    # Step 3: Fall back to ESMFold
    if len(seq) > 400:
        print(f'\n  WARNING: {len(seq)} aa — ESMFold may be slow or fail above ~400 aa.')
    print('Running ESMFold...', end=' ', flush=True)
    try:
        run_esmfold(seq, acc)
        structure_source[acc] = 'ESMFold'
        kb = os.path.getsize(struct_path(acc)) / 1024
        print(f'done ({kb:.0f} KB).')
    except Exception as e:
        print(f'FAILED: {e}')
        structure_source[acc] = 'FAILED'

# Summary
print(f'\n{SEP}')
print('STRUCTURE SUMMARY')
print(SEP)
for acc in ALL_ACCS:
    exists = 'OK     ' if os.path.exists(struct_path(acc)) else 'MISSING'
    src    = structure_source.get(acc, '?')
    print(f'  {exists}  [{src:10s}]  {acc}')

STEPS 1-3: FETCHING SEQUENCES AND STRUCTURES

[YP_009168097.1]
  NCBI: 132 aa — YP_009168097.1 DUF1627 domain-containing protein [Escherichia phage vB
  AlphaFold DB: mapping to UniProt... UniProt=G3CFH0, checking DB... not in DB. Running ESMFold... done (84 KB).

[WP_000994792.1]
  NCBI: 132 aa — WP_000994792.1 DUF1627 domain-containing protein [Escherichia coli]
  AlphaFold DB: mapping to UniProt... no UniProt mapping. Running ESMFold... done (84 KB).

[YP_009168179.1]
  NCBI: 126 aa — YP_009168179.1 hypothetical protein APL45_gp45 [Escherichia phage vB_E
  AlphaFold DB: mapping to UniProt... UniProt=G3CFQ2, checking DB... downloaded.

[KLD66510.1]
  NCBI: 85 aa — KLD66510.1 hypothetical protein Y886_44650, partial [Xanthomonas hyaci
  AlphaFold DB: mapping to UniProt... no UniProt mapping. Running ESMFold... done (56 KB).

[YP_009168137.1]
  NCBI: 219 aa — YP_009168137.1 type A-1 chloramphenicol O-acetyltransferase [Cloning v
  AlphaFold DB: mapping to UniProt... UniProt=G3CFL0, che

In [ ]:
# Step 4: TM-score and RMSD for each HGT candidate pair

SEP = '=' * 65
print(SEP)
print('STEP 4: STRUCTURAL COMPARISON (TMalign)')
print(SEP)

tm_results = {}  # (phage_acc, bact_acc) -> (tm, rmsd, aln_len)

if not os.path.exists('./TMalign'):
    print('TMalign binary not found.')
    print('Manual alternative: https://zhanggroup.org/US-align/')
    print('Upload both .pdb files, run alignment, record TM-score and RMSD.')
else:
    for phage_acc, bact_acc, row_label, blast_note in PAIRS:
        p1 = struct_path(phage_acc)
        p2 = struct_path(bact_acc)
        print(f'\n{row_label}: {phage_acc}  vs  {bact_acc}')

        if not os.path.exists(p1):
            print(f'  SKIP — missing phage structure: {p1}')
            continue
        if not os.path.exists(p2):
            print(f'  SKIP — missing bacterial structure: {p2}')
            continue

        # -o writes superimposed PDBs for visualization
        out_prefix = f'results/superimposed_{slug(phage_acc)}_vs_{slug(bact_acc)}'
        tm, rmsd, aln_len = run_tmalign(p1, p2, out_prefix=out_prefix)
        tm_results[(phage_acc, bact_acc)] = (tm, rmsd, aln_len)

        print(f'  TM-score : {tm:.4f}' if tm   else '  TM-score : N/A')
        print(f'  RMSD     : {rmsd:.2f} Å' if rmsd else '  RMSD     : N/A')
        print(f'  Aln len  : {aln_len} residues' if aln_len else '')
        print(f'  Result   : {tm_label(tm)}')

STEP 4: STRUCTURAL COMPARISON (TMalign)

Row 1: YP_009168097.1  vs  WP_000994792.1
  TM-score : 1.0000
  RMSD     : N/A
  Aln len  : 132 residues
  Result   : STRONG structural similarity — supports HGT

Row 2: YP_009168179.1  vs  KLD66510.1
  TM-score : 0.2323
  RMSD     : 4.31 Å
  Aln len  : 36 residues
  Result   : WEAK structural similarity — inconclusive

Row 3: YP_009168137.1  vs  EDS05563
  TM-score : 0.9982
  RMSD     : 0.26 Å
  Aln len  : 219 residues
  Result   : STRONG structural similarity — supports HGT

Row 10: YP_009168122.1  vs  EFQ7096440.1
  SKIP — missing phage structure: structures/YP_009168122_1.pdb

Row 11: YP_009168121.1  vs  WP_000062360.1
  TM-score : 1.0000
  RMSD     : N/A
  Aln len  : 258 residues
  Result   : STRONG structural similarity — supports HGT


In [ ]:
# Step 5: Foldseek — search phage proteins against AlphaFold DB + PDB
# This is independent structural corroboration: if bacterial hits top the results,
# that is strong evidence of structural conservation consistent with HGT.

SEP = '=' * 65
print(SEP)
print('STEP 5: FOLDSEEK STRUCTURAL DATABASE SEARCH')
print('Phage proteins searched against AlphaFold DB (afdb50) + PDB')
print(SEP)

foldseek_hits = {}  # phage_acc -> list of hit dicts

for phage_acc, bact_acc, row_label, blast_note in PAIRS:
    pdb = struct_path(phage_acc)
    print(f'\n{row_label}: {phage_acc}')
    if not os.path.exists(pdb):
        print('  SKIP — structure missing.')
        continue
    print('  Submitting to Foldseek...', end=' ', flush=True)
    try:
        hits = run_foldseek(pdb)
        foldseek_hits[phage_acc] = hits
        print(f'{len(hits)} hits returned.')
        for h in hits[:5]:
            print(f'    [{h["db"]:12s}]  TM={h["tm_score"]:.4f}  {h["target"]:30s}  {h["organism"]}')
    except Exception as e:
        print(f'FAILED: {e}')
        foldseek_hits[phage_acc] = []

STEP 5: FOLDSEEK STRUCTURAL DATABASE SEARCH
Phage proteins searched against AlphaFold DB (afdb50) + PDB

Row 1: YP_009168097.1
  Submitting to Foldseek... FAILED: 'list' object has no attribute 'get'

Row 2: YP_009168179.1
  Submitting to Foldseek... FAILED: 'list' object has no attribute 'get'

Row 3: YP_009168137.1
  Submitting to Foldseek... FAILED: 'list' object has no attribute 'get'

Row 10: YP_009168122.1
  SKIP — structure missing.

Row 11: YP_009168121.1
  Submitting to Foldseek... FAILED: 'list' object has no attribute 'get'


In [ ]:
# Step 6: Generate manuscript results table

SEP = '=' * 65
print(SEP)
print('STEP 6: MANUSCRIPT RESULTS TABLE')
print(SEP)

rows = []
for phage_acc, bact_acc, row_label, blast_note in PAIRS:
    tm, rmsd, aln_len = tm_results.get((phage_acc, bact_acc), (None, None, None))

    # Top Foldseek hits (bacterial, if any)
    fs_hits = foldseek_hits.get(phage_acc, [])
    top_fs  = '; '.join(f"{h['target']} (TM={h['tm_score']:.3f}, {h['organism']})" for h in fs_hits[:2])

    rows.append({
        'Row':                       row_label,
        'Phage Accession':           phage_acc,
        'Bacterial Accession':       bact_acc,
        'BLAST Evidence':            blast_note,
        'Phage Structure Source':    structure_source.get(phage_acc, 'N/A'),
        'Bacterial Structure Source':structure_source.get(bact_acc,  'N/A'),
        'TM-score':                  f'{tm:.4f}' if tm      else 'N/A',
        'RMSD (Angstrom)':           f'{rmsd:.2f}'  if rmsd  else 'N/A',
        'Aligned Residues':          aln_len if aln_len      else 'N/A',
        'Structural Interpretation': tm_label(tm),
        'Top Foldseek Hits':         top_fs if top_fs        else 'N/A',
    })

df = pd.DataFrame(rows)
csv_path = 'results/hgt_structural_validation.csv'
df.to_csv(csv_path, index=False)

# Display with key columns only
display_cols = ['Row', 'Phage Accession', 'Bacterial Accession',
                'TM-score', 'RMSD (Angstrom)', 'Structural Interpretation']
print(df[display_cols].to_string(index=False))
print(f'\nFull table saved: {csv_path}')

STEP 6: MANUSCRIPT RESULTS TABLE
   Row Phage Accession Bacterial Accession TM-score RMSD (Angstrom)                      Structural Interpretation
 Row 1  YP_009168097.1      WP_000994792.1   1.0000             N/A    STRONG structural similarity — supports HGT
 Row 2  YP_009168179.1          KLD66510.1   0.2323            4.31      WEAK structural similarity — inconclusive
 Row 3  YP_009168137.1            EDS05563   0.9982            0.26    STRONG structural similarity — supports HGT
Row 10  YP_009168122.1        EFQ7096440.1      N/A             N/A N/A — structure missing or TMalign unavailable
Row 11  YP_009168121.1      WP_000062360.1   1.0000             N/A    STRONG structural similarity — supports HGT

Full table saved: results/hgt_structural_validation.csv


In [ ]:
# Step 7: 3D superposition visualization
# Blue = phage protein, Red = bacterial protein
# Uses superimposed PDB files output by TMalign (-o flag) for accurate overlay.

import py3Dmol
from IPython.display import display as ipy_display

MAX_PAIRS_TO_SHOW = 5  # change to show fewer/more

shown = 0
for phage_acc, bact_acc, row_label, blast_note in PAIRS:
    if shown >= MAX_PAIRS_TO_SHOW:
        break

    # Prefer TMalign-superimposed structures; fall back to raw
    sup_prefix = f'results/superimposed_{slug(phage_acc)}_vs_{slug(bact_acc)}'
    p1_vis = f'{sup_prefix}_all_atm' if os.path.exists(f'{sup_prefix}_all_atm') else struct_path(phage_acc)
    p2_vis = f'{sup_prefix}_all_atm_b' if os.path.exists(f'{sup_prefix}_all_atm_b') else struct_path(bact_acc)

    if not os.path.exists(struct_path(phage_acc)) or not os.path.exists(struct_path(bact_acc)):
        print(f'{row_label}: missing structure — skipping visualization.')
        continue

    tm_val = tm_results.get((phage_acc, bact_acc), (None,))[0]
    tm_str = f'TM-score={tm_val:.4f}' if tm_val else 'TM-score=N/A'
    print(f'\n{row_label}: {phage_acc} (blue)  vs  {bact_acc} (red)  |  {tm_str}')

    with open(struct_path(phage_acc)) as f: pdb1 = f.read()
    with open(struct_path(bact_acc))  as f: pdb2 = f.read()

    view = py3Dmol.view(width=820, height=460)
    view.addModel(pdb1, 'pdb')
    view.setStyle({'model': 0}, {'cartoon': {'color': 'steelblue', 'opacity': 0.9}})
    view.addModel(pdb2, 'pdb')
    view.setStyle({'model': 1}, {'cartoon': {'color': 'firebrick', 'opacity': 0.9}})
    view.zoomTo()
    view.show()

    shown += 1

print('\nFor poster figures: screenshot each panel.')
print('For proper superimposition: use Mol* at https://molstar.org/viewer')
print('  -> Open Files -> load both PDBs -> Superpose')


Row 1: YP_009168097.1 (blue)  vs  WP_000994792.1 (red)  |  TM-score=1.0000


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


Row 2: YP_009168179.1 (blue)  vs  KLD66510.1 (red)  |  TM-score=0.2323


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


Row 3: YP_009168137.1 (blue)  vs  EDS05563 (red)  |  TM-score=0.9982


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Row 10: missing structure — skipping visualization.

Row 11: YP_009168121.1 (blue)  vs  WP_000062360.1 (red)  |  TM-score=1.0000


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


For poster figures: screenshot each panel.
For proper superimposition: use Mol* at https://molstar.org/viewer
  -> Open Files -> load both PDBs -> Superpose


In [ ]:
# Download all outputs to your computer

from google.colab import files
import glob

print('Downloading files...')

# PDB structure files
for acc in ALL_ACCS:
    path = struct_path(acc)
    if os.path.exists(path):
        files.download(path)
        print(f'  {path} [{structure_source.get(acc, "?")}]')

# Results table
if os.path.exists('results/hgt_structural_validation.csv'):
    files.download('results/hgt_structural_validation.csv')
    print('  results/hgt_structural_validation.csv')

# Superimposed structures for visualization
for f_path in glob.glob('results/superimposed_*.pdb'):
    files.download(f_path)
    print(f'  {f_path}')

print('\nAll done.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  structures/YP_009168097_1.pdb [ESMFold]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

  results/superimposed_YP_009168097_1_vs_WP_000994792_1.pdb


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  results/superimposed_YP_009168137_1_vs_EDS05563.pdb

All done.
